# 第5章 単層ネットワーク: 分類 (Single-layer Networks: Classification)
## 5.1 判別関数 (Discriminant Functions)

### 本ノートブックの目的と概要
本ノートブックでは、**Christopher M. Bishop & Hugh Bishop (2024) 『Deep Learning: Foundations and Concepts』第5章「Single-layer Networks: Classification」第1節「5.1 Discriminant Functions」** を完全に網羅し、分類問題の基礎となる線形判別関数の理論的導出、幾何学的性質、多クラスへの拡張、1-of-K 符号化、および分類への最小二乗法の適用と外れ値に対する脆弱性を、数式・Python実装・教科書図版（Figure 5.1 〜 5.4）の完全再現によって探究します。

---

### セクション構成
- **5.1.1 2クラス判別関数 (Two Classes)**:
  - 線形判別モデル $y(\mathbf{x}) = \mathbf{w}^T \mathbf{x} + w_0$
  - 法線ベクトル $\mathbf{w}$ の直交性証明
  - 原点からの直交距離 $-w_0 / \|\mathbf{w}\|$
  - 任意の点 $\mathbf{x}$ からの符号付き直交距離 $r = y(\mathbf{x}) / \|\mathbf{w}\|$ および直交射影 $\mathbf{x}_\perp$
  - 拡張表現 $y(\mathbf{x}) = \widetilde{\mathbf{w}}^T \widetilde{\mathbf{x}}$
  - **Figure 5.1**: 2次元における線形判別関数の幾何学的構造の完全再現
- **5.1.2 多クラス判別関数 (Multiple Classes)**:
  - 2クラス分類器の組み合わせ（ヒューリスティック手法）の破綻:
    - 1対他 (1-versus-the-rest) 分類器と曖昧領域
    - 1対1 (1-versus-1) 分類器と曖昧領域
  - **Figure 5.2**: 3クラス問題における曖昧領域（緑色領域 `?`）の完全再現
  - 単一の $K$ クラス線形判別関数 $y_k(\mathbf{x}) = \mathbf{w}_k^T \mathbf{x} + w_{k0}$
  - 決定境界 $(\mathbf{w}_k - \mathbf{w}_j)^T \mathbf{x} + (w_{k0} - w_{j0}) = 0$
  - 決定領域 $\mathcal{R}_k$ の凸性 (Convexity) と単連結性の厳密な数学的証明
  - **Figure 5.3**: 決定領域の凸性を示す図解の完全再現
- **5.1.3 1-of-K 符号化 (1-of-K Coding)**:
  - 2クラスのバイナリラベル $t \in \{0, 1\}$
  - $K$ クラスのワンホットベクトル $\mathbf{t} = (0, \dots, 1, \dots, 0)^T$
  - 確率的解釈 $t_k = p(\mathcal{C}_k)$ の限界
- **5.1.4 分類のための最小二乗法 (Least Squares for Classification)**:
  - 行列形式と二乗誤差関数 $E_D(\widetilde{\mathbf{W}})$
  - 正規方程式と閉形式解 $\widetilde{\mathbf{W}} = \widetilde{\mathbf{X}}^\dagger \mathbf{T}$
  - ターゲットの線形制約保存定理の証明: $\mathbf{a}^T \mathbf{t}_n + b = 0 \implies \mathbf{a}^T \mathbf{y}(\mathbf{x}) + b = 0$
  - 最小二乗分類の根本的欠陥（確率の公理違反、ガウス仮定の不整合、外れ値への極端な感度）
  - **Figure 5.4**: 外れ値に対する最小二乗法（マゼンタ線）の脆弱性とロジスティック回帰（緑線）の頑健性の完全再現


In [ ]:
import sys
import os
sys.path.append("..")

import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt

# モジュールインポート
from common.discriminant_functions import (
    LinearDiscriminant2Class,
    LinearDiscriminantMultiClass,
    OneVersusRestClassifier,
    OneVersusOneClassifier,
    LeastSquaresClassifier,
    LogisticRegression2Class,
    to_one_of_k,
    plot_figure_5_1_geometry,
    plot_figure_5_2_ambiguities,
    plot_figure_5_3_convex_regions,
    plot_figure_5_4_least_squares_outliers,
)

print("Modules imported successfully!")


---
## 5.1.1 2クラス判別関数 (Two Classes)

### 1. 数学モデルと決定境界
判別関数（discriminant function）とは、入力ベクトル $\mathbf{x} \in \mathbb{R}^D$ を直接いずれかの離散クラス $\mathcal{C}_k$ に割り当てる関数です。
最も単純な線形判別関数は、入力変数の線形結合として次のように定義されます：

$$
y(\mathbf{x}) = \mathbf{w}^T \mathbf{x} + w_0 \tag{5.2}
$$

ここで、$\mathbf{w}$ は重みベクトル（weight vector）、$w_0$ はバイアス（bias: 統計学的な推定量の偏りとは異なるニューラルネットワーク用語）です。
分類規則は次の通りです：
- $y(\mathbf{x}) \ge 0 \implies \mathbf{x} \in \mathcal{C}_1$
- $y(\mathbf{x}) < 0 \implies \mathbf{x} \in \mathcal{C}_2$

決定境界（decision boundary / decision surface）は $y(\mathbf{x}) = 0$ によって定義され、これは $D$ 次元入力空間における $(D-1)$ 次元の超平面（hyperplane）となります。

---

### 2. 幾何学的性質の厳密な導出

#### (1) 法線ベクトル $\mathbf{w}$ の直交性
決定境界上の任意の2点 $\mathbf{x}_A, \mathbf{x}_B$ をとると、$y(\mathbf{x}_A) = 0$ かつ $y(\mathbf{x}_B) = 0$ が成り立ちます。
これらを差し引くと：
$$
y(\mathbf{x}_A) - y(\mathbf{x}_B) = (\mathbf{w}^T \mathbf{x}_A + w_0) - (\mathbf{w}^T \mathbf{x}_B + w_0) = \mathbf{w}^T (\mathbf{x}_A - \mathbf{x}_B) = 0
$$
$\mathbf{x}_A - \mathbf{x}_B$ は決定境界面内に平行な任意のベクトルを表すため、重みベクトル $\mathbf{w}$ は境界面内のあらゆるベクトルと直交します。
すなわち、**$\mathbf{w}$ は決定境界の法線ベクトル（normal vector）であり、境界の向き（orientation）を決定**します。

#### (2) 原点からの直交距離
原点 $\mathbf{0}$ から決定境界へ下ろした垂線の足を $\mathbf{x}_0$ とすると、$\mathbf{x}_0$ は法線方向の単位ベクトル $\frac{\mathbf{w}}{\|\mathbf{w}\|}$ に平行なので、あるスカラー $d$ を用いて $\mathbf{x}_0 = d \frac{\mathbf{w}}{\|\mathbf{w}\|}$ と表せます。
$\mathbf{x}_0$ は境界上にあるため $y(\mathbf{x}_0) = 0$：
$$
\mathbf{w}^T \left( d \frac{\mathbf{w}}{\|\mathbf{w}\|} \right) + w_0 = d \|\mathbf{w}\| + w_0 = 0 \implies d = -\frac{w_0}{\|\mathbf{w}\|} \tag{5.3}
$$
したがって、**バイアスパラメータ $w_0$ は決定境界の原点からの位置（変位）を制御**します。

#### (3) 任意の点 $\mathbf{x}$ からの符号付き直交距離
空間内の任意の点 $\mathbf{x}$ について、決定境界への直交射影点を $\mathbf{x}_\perp$ とし、垂直距離を $r$ と置くと：
$$
\mathbf{x} = \mathbf{x}_\perp + r \frac{\mathbf{w}}{\|\mathbf{w}\|} \tag{5.4}
$$
両辺に $\mathbf{w}^T$ を掛け、さらに $w_0$ を加えると：
$$
\mathbf{w}^T \mathbf{x} + w_0 = \mathbf{w}^T \mathbf{x}_\perp + w_0 + r \frac{\mathbf{w}^T \mathbf{w}}{\|\mathbf{w}\|}
$$
ここで左辺は $y(\mathbf{x})$ であり、また $\mathbf{x}_\perp$ は境界上にあるため $\mathbf{w}^T \mathbf{x}_\perp + w_0 = y(\mathbf{x}_\perp) = 0$ です。右辺第2項は $r \|\mathbf{w}\|$ となるので：
$$
y(\mathbf{x}) = r \|\mathbf{w}\| \implies r = \frac{y(\mathbf{x})}{\|\mathbf{w}\|} \tag{5.5}
$$
すなわち、**判別関数の値 $y(\mathbf{x})$ は、点 $\mathbf{x}$ から決定境界までの符号付き直交距離（signed orthogonal distance）に比例**します。

#### (4) 拡張表現 (Augmented Input)
ダミー入力 $x_0 = 1$ を導入し、
$$
\widetilde{\mathbf{w}} = (w_0, \mathbf{w}^T)^T, \quad \widetilde{\mathbf{x}} = (1, \mathbf{x}^T)^T
$$
と定義すると、判別関数は簡潔に内積で表されます：
$$
y(\mathbf{x}) = \widetilde{\mathbf{w}}^T \widetilde{\mathbf{x}} \tag{5.6}
$$
このとき決定境界面は、$(D+1)$ 次元の拡張入力空間において原点を通る $D$ 次元の超平面となります。


In [ ]:
# Figure 5.1: Geometry of a linear discriminant function in two dimensions
fig5_1 = plot_figure_5_1_geometry(filepath="result/fig_5_1_discriminant_geometry.png")
plt.show()

# 幾何学的性質の数値シミュレーション検証
w = np.array([0.75, 1.0])
w0 = -3.5
disc = LinearDiscriminant2Class(w, w0)

# (1) 決定境界上の2点間の直交性検証
xA = np.array([2.0, -(w[0]*2.0 + w0)/w[1]])
xB = np.array([4.0, -(w[0]*4.0 + w0)/w[1]])
assert np.isclose(disc.decision_function(xA), 0.0)
assert np.isclose(disc.decision_function(xB), 0.0)
assert np.isclose(np.dot(w, xA - xB), 0.0), "w must be orthogonal to vectors in the boundary!"

# (2) 原点からの直交距離の検証
assert np.isclose(disc.normal_distance_from_origin(), -w0 / np.linalg.norm(w))

# (3) 任意点の直交射影と符号付き距離の検証
x_test = np.array([5.0, 3.2])
r = disc.margin(x_test)
x_perp = disc.orthogonal_projection(x_test)
assert np.isclose(disc.decision_function(x_perp), 0.0), "Projection must lie on the decision boundary!"
assert np.isclose(np.linalg.norm(x_test - x_perp), np.abs(r))
print(f"Point x: {x_test}")
print(f"Signed distance r = y(x)/||w||: {r:.4f}")
print(f"Orthogonal projection x_perp: {x_perp}")
print("Geometric assertions passed 100%!")


---
## 5.1.2 多クラス判別関数 (Multiple Classes)

$K > 2$ クラスの分類問題に線形判別関数を拡張する場合を考えます。
まず思いつくのは、2クラス判別関数を組み合わせて多クラス分類を行うヒューリスティックな手法ですが、これらは深刻な未分類・曖昧領域（ambiguous regions）を生み出します。

### 1. ヒューリスティックな組み合わせの破綻

#### (1) 1対他 (One-versus-the-rest) 分類器
$K-1$ 個の2クラス分類器を用意し、各分類器 $k$ が「クラス $\mathcal{C}_k$ に属するか、それ以外か」を判定します。
- 致命的欠陥: 入力空間の一部で、複数の分類器が同時に「所属する（$y_k > 0$）」と判定したり、どの分類器も所属しない（全て $y_k < 0$）と判定する領域が生じます。
- Figure 5.2（左図）の緑色領域（`?` マーク）に示すように、曖昧に分類される領域（ambiguously classified region）が必然的に発生します。

#### (2) 1対1 (One-versus-one) 分類器
可能なすべてのクラスペアに対して 1 つずつ、合計 $\frac{K(K-1)}{2}$ 個の2クラス分類器を用意し、多数決（majority voting）によってクラスを決定します。
- 致命的欠陥: 3クラス問題において、$\mathcal{C}_1 > \mathcal{C}_2$, $\mathcal{C}_2 > \mathcal{C}_3$, $\mathcal{C}_3 > \mathcal{C}_1$ という巡回矛盾（引き分け）が発生する領域が存在します。
- Figure 5.2（右図）の中央の緑色三角形領域（`?` マーク）に示すように、各クラスへの投票が1票ずつで拮抗し、一意に決定できない曖昧領域が生じます。

---

### 2. 単一の $K$ クラス判別関数
これらの困難は、$K$ 個の線形関数からなる単一の多クラス判別関数を用いることで完全に解決されます：

$$
y_k(\mathbf{x}) = \mathbf{w}_k^T \mathbf{x} + w_{k0} \tag{5.7}
$$

入力 $\mathbf{x}$ は、$y_k(\mathbf{x}) > y_j(\mathbf{x}) \quad (\forall j \neq k)$ を満たすクラス $\mathcal{C}_k$ に割り当てられます。
このとき、クラス $\mathcal{C}_k$ と $\mathcal{C}_j$ の決定境界は $y_k(\mathbf{x}) = y_j(\mathbf{x})$ で与えられ、
$$
(\mathbf{w}_k - \mathbf{w}_j)^T \mathbf{x} + (w_{k0} - w_{j0}) = 0 \tag{5.8}
$$
となり、2クラスの場合と全く同様に $(D-1)$ 次元の超平面となります。

---

### 3. 決定領域 $\mathcal{R}_k$ の凸性 (Convexity) と単連結性の厳密な数学的証明

この多クラス線形判別関数によって分割される各決定領域 $\mathcal{R}_k$ は、**常に単連結（singly connected）かつ凸（convex）** になります。

**【証明】**
領域 $\mathcal{R}_k$ に属する任意の2点 $\mathbf{x}_A, \mathbf{x}_B \in \mathcal{R}_k$ をとります。
この2点を結ぶ線分上の任意の点 $\widehat{\mathbf{x}}$ は、ある $0 \le \lambda \le 1$ を用いて次のように表されます：
$$
\widehat{\mathbf{x}} = \lambda \mathbf{x}_A + (1 - \lambda) \mathbf{x}_B \tag{5.9}
$$
判別関数 $y_k(\mathbf{x}) = \mathbf{w}_k^T \mathbf{x} + w_{k0}$ は入力 $\mathbf{x}$ について線形（アフィン変換）であるため、内分点における値は：
$$
y_k(\widehat{\mathbf{x}}) = \lambda y_k(\mathbf{x}_A) + (1 - \lambda) y_k(\mathbf{x}_B) \tag{5.10}
$$
ここで、$\mathbf{x}_A, \mathbf{x}_B \in \mathcal{R}_k$ であることから、任意の $j \neq k$ に対して
$$
y_k(\mathbf{x}_A) > y_j(\mathbf{x}_A) \quad \text{かつ} \quad y_k(\mathbf{x}_B) > y_j(\mathbf{x}_B)
$$
が成り立ちます。したがって、$\lambda \ge 0$ かつ $1 - \lambda \ge 0$（かつ両者が同時に0にはならない）より、
$$
y_k(\widehat{\mathbf{x}}) = \lambda y_k(\mathbf{x}_A) + (1 - \lambda) y_k(\mathbf{x}_B) > \lambda y_j(\mathbf{x}_A) + (1 - \lambda) y_j(\mathbf{x}_B) = y_j(\widehat{\mathbf{x}})
$$
がすべての $j \neq k$ に対して成立します。
したがって $\widehat{\mathbf{x}} \in \mathcal{R}_k$ であり、線分上のすべての点が領域 $\mathcal{R}_k$ 内に含まれます。
ゆえに、決定領域 $\mathcal{R}_k$ は凸集合であり、自動的に単連結（穴や飛び地が存在しない）です。 $\blacksquare$


In [ ]:
# Figure 5.2: Ambiguous regions in heuristic multi-class classifiers
fig5_2 = plot_figure_5_2_ambiguities(filepath="result/fig_5_2_multiclass_ambiguities.png")
plt.show()

# Figure 5.3: Convex decision regions of multi-class linear discriminant
fig5_3 = plot_figure_5_3_convex_regions(filepath="result/fig_5_3_convex_regions.png")
plt.show()

# 凸性の数値シミュレーション検証
W = np.array([
    [-1.0, 1.2],  # class 0
    [1.0, 1.2],   # class 1
    [0.0, -1.5],  # class 2
])
W0 = np.array([0.0, 0.0, 0.5])
multi_disc = LinearDiscriminantMultiClass(W, W0)

# 領域 R_2 内の2点
xA = np.array([-1.5, -2.0])
xB = np.array([1.5, -2.0])
assert multi_disc.predict(xA) == 2
assert multi_disc.predict(xB) == 2

# 線分上の100点をサンプリングしてすべてクラス2に判定されることを確認
for lam in np.linspace(0.0, 1.0, 100):
    x_hat = lam * xA + (1.0 - lam) * xB
    assert multi_disc.predict(x_hat) == 2, f"Convexity violated at lambda={lam}!"

print("Multi-class convex region assertions passed 100%!")


---
## 5.1.3 1-of-K 符号化 (1-of-K Coding)

分類問題におけるターゲット変数（教師ラベル）の表現形式を定義します。

### 1. 2クラス問題のバイナリ表現
2クラス問題では、スカラー変数 $t \in \{0, 1\}$ を用いて表現するのが最も自然です：
- $t = 1 \implies \mathbf{x} \in \mathcal{C}_1$
- $t = 0 \implies \mathbf{x} \in \mathcal{C}_2$
このとき、$t$ の値はクラスが $\mathcal{C}_1$ である確率 $p(\mathcal{C}_1)$ と解釈することができます。

### 2. $K$ クラス問題の 1-of-K 符号化 (One-Hot Encoding)
$K > 2$ クラス問題では、長さ $K$ のバイナリベクトル $\mathbf{t}$ を導入します。観測がクラス $\mathcal{C}_j$ に属する場合、$j$ 番目の要素 $t_j = 1$ とし、それ以外のすべての要素 $t_k = 0 \; (k \neq j)$ とします。
例えば $K = 5$ クラスで、クラス 2 に属するデータ点のターゲットベクトルは：
$$
\mathbf{t} = (0, 1, 0, 0, 0)^T \tag{5.11}
$$
このとき各要素 $t_k$ は、データがクラス $\mathcal{C}_k$ である確率 $p(\mathcal{C}_k)$ の実現値と解釈できます。


In [ ]:
# 1-of-K 符号化の動作確認
labels = np.array([0, 1, 2, 1, 0, 4])
T_onehot = to_one_of_k(labels, K=5)
print("Class labels:", labels)
print("1-of-K Encoded Target Matrix T:")
print(T_onehot)

# 各行の和が常に 1 であることを検証
assert np.all(T_onehot.sum(axis=1) == 1.0)
print("1-of-K encoding verified successfully!")


---
## 5.1.4 分類のための最小二乗法 (Least Squares for Classification)

線形回帰において二乗和誤差関数の最小化が単純な解析的閉形式解（closed-form solution）を与えたことから、分類問題にも最小二乗法の枠組みを適用することが検討されます。

### 1. 行列形式と二乗和誤差関数
各クラス $\mathcal{C}_k$ の線形モデル $y_k(\mathbf{x}) = \mathbf{w}_k^T \mathbf{x} + w_{k0}$ を、拡張ベクトル $\widetilde{\mathbf{x}} = (1, \mathbf{x}^T)^T$ と $\widetilde{\mathbf{w}}_k = (w_{k0}, \mathbf{w}_k^T)^T$ を用いてまとめます：
$$
\mathbf{y}(\mathbf{x}) = \widetilde{\mathbf{W}}^T \widetilde{\mathbf{x}} \tag{5.13}
$$
ここで $\widetilde{\mathbf{W}}$ は $k$ 列目に $\widetilde{\mathbf{w}}_k$ を持つ $(D+1) \times K$ の重み行列です。

訓練データセット $\{\mathbf{x}_n, \mathbf{t}_n\}_{n=1}^N$ に対し、$\mathbf{t}_n^T$ を $n$ 行目に持つ $N \times K$ 行列 $\mathbf{T}$ と、$\widetilde{\mathbf{x}}_n^T$ を $n$ 行目に持つ $N \times (D+1)$ 行列 $\widetilde{\mathbf{X}}$ を定義すると、二乗和誤差関数はトレースを用いて次のように書けます：
$$
E_D(\widetilde{\mathbf{W}}) = \frac{1}{2} \mathrm{Tr}\left\{ (\widetilde{\mathbf{X}} \widetilde{\mathbf{W}} - \mathbf{T})^T (\widetilde{\mathbf{X}} \widetilde{\mathbf{W}} - \mathbf{T}) \right\} \tag{5.14}
$$

### 2. 閉形式解 (Closed-form Solution)
$\widetilde{\mathbf{W}}$ に関する微分を 0 と置くことで、正規方程式が得られます：
$$
\widetilde{\mathbf{X}}^T (\widetilde{\mathbf{X}} \widetilde{\mathbf{W}} - \mathbf{T}) = \mathbf{0}
$$
これを解くと、ムーア・ペンローズ擬似逆行列 $\widetilde{\mathbf{X}}^\dagger$ を用いてパラメータ行列が求まります：
$$
\widetilde{\mathbf{W}} = (\widetilde{\mathbf{X}}^T \widetilde{\mathbf{X}})^{-1} \widetilde{\mathbf{X}}^T \mathbf{T} = \widetilde{\mathbf{X}}^\dagger \mathbf{T} \tag{5.15}
$$
未知の入力 $\mathbf{x}$ に対する予測値は次のように与えられます：
$$
\mathbf{y}(\mathbf{x}) = \widetilde{\mathbf{W}}^T \widetilde{\mathbf{x}} = \mathbf{T}^T (\widetilde{\mathbf{X}}^\dagger)^T \widetilde{\mathbf{x}} \tag{5.16}
$$

---

### 3. ターゲットの線形制約保存定理 (Linear Constraint Preservation)

多変数最小二乗法の重要な性質として、訓練セット内のすべてのターゲットベクトル $\mathbf{t}_n$ がある線形制約
$$
\mathbf{a}^T \mathbf{t}_n + b = 0 \tag{5.17}
$$
（$\mathbf{a}, b$ は定数）を満たすならば、モデルの任意の入力 $\mathbf{x}$ に対する予測値 $\mathbf{y}(\mathbf{x})$ も全く同一の線形制約
$$
\mathbf{a}^T \mathbf{y}(\mathbf{x}) + b = 0 \tag{5.18}
$$
を満たします。

**【証明】**
訓練セットの全データで制約が成り立つので、全要素が 1 のベクトル $\mathbf{1} = (1, \dots, 1)^T$ を用いて $\mathbf{T} \mathbf{a} + b \mathbf{1} = \mathbf{0}$ と書けます。
拡張入力行列 $\widetilde{\mathbf{X}}$ の第 1 列はすべて 1 であるため、$\mathbf{e}_1 = (1, 0, \dots, 0)^T$ と置くと $\widetilde{\mathbf{X}} \mathbf{e}_1 = \mathbf{1}$ です。
したがって、
$$
\widetilde{\mathbf{W}} \mathbf{a} = (\widetilde{\mathbf{X}}^T \widetilde{\mathbf{X}})^{-1} \widetilde{\mathbf{X}}^T \mathbf{T} \mathbf{a} = (\widetilde{\mathbf{X}}^T \widetilde{\mathbf{X}})^{-1} \widetilde{\mathbf{X}}^T (-b \mathbf{1}) = -b (\widetilde{\mathbf{X}}^T \widetilde{\mathbf{X}})^{-1} \widetilde{\mathbf{X}}^T (\widetilde{\mathbf{X}} \mathbf{e}_1) = -b \mathbf{e}_1
$$
この結果を任意の入力 $\mathbf{x}$ の予測に適用すると：
$$
\mathbf{a}^T \mathbf{y}(\mathbf{x}) = (\widetilde{\mathbf{W}} \mathbf{a})^T \widetilde{\mathbf{x}} = (-b \mathbf{e}_1)^T \widetilde{\mathbf{x}} = -b \cdot 1 = -b
$$
ゆえに、$\mathbf{a}^T \mathbf{y}(\mathbf{x}) + b = 0$ が常に成立します。 $\blacksquare$

この定理から、1-of-K 符号化（$\sum_k t_{nk} = 1 \iff \mathbf{1}^T \mathbf{t}_n - 1 = 0$）を採用した場合、**任意の入力 $\mathbf{x}$ に対するモデル出力の総和は常に 1**（$\sum_k y_k(\mathbf{x}) = 1$）になります。

---

### 4. 最小二乗分類の根本的欠陥と外れ値に対する脆弱性

上記の好ましい性質にもかかわらず、分類に対する最小二乗法は実用上極めて深刻な問題を抱えています：

1. **確率の公理違反**:
   総和が 1 になる制約は満たしても、個々の出力 $y_k(\mathbf{x})$ が区間 $[0, 1]$ 内に収まる保証は全くありません。負の値や 1 を超える値を出力するため、厳密な事後確率として解釈できません。
2. **ガウス雑音仮定の不整合**:
   二乗和誤差の最小化は、目標値がガウス分布に従うという最尤推定に相当します。しかし、離散的なバイナリターゲット $t \in \{0, 1\}$ は明らかにガウス分布とはかけ離れています。
3. **外れ値 (Outliers) に対する極端な脆弱性**:
   二乗誤差 $(y_k(\mathbf{x}) - t_k)^2$ は、決定境界から正しく分類された遠方の点に対しても、予測値が 1 よりはるかに大きくなるため「誤差が大きい」と判定して過剰なペナルティを課します。
   その結果、**決定境界が正しいクラスの遠方データ（外れ値）の方向へ無理に引き寄せられ、本来正しく分類されていたデータ群が大量に誤分類されてしまいます**。

これに対し、ロジスティック回帰（負の対数尤度 / クロスエントロピー誤差）は確率を出力し、正しい側に十分離れたデータポイントにはペナルティが漸近的に 0 になるため、外れ値に対して極めて頑健（robust）です。


In [ ]:
# Figure 5.4: Least squares vs logistic regression sensitivity to outliers
fig5_4 = plot_figure_5_4_least_squares_outliers(filepath="result/fig_5_4_least_squares_outliers.png")
plt.show()

# 最小二乗法とロジスティック回帰の誤分類率の定量的比較
rng = np.random.RandomState(42)
N1, N2 = 30, 45
X1 = rng.multivariate_normal([-0.5, 1.5], [[1.0, 0.7], [0.7, 1.0]], size=N1)
X2 = rng.multivariate_normal([0.9, -1.2], [[0.8, 0.6], [0.6, 1.1]], size=N2)
X_clean = np.vstack([X1, X2])
y_clean = np.hstack([np.zeros(N1), np.ones(N2)])
T_clean = to_one_of_k(y_clean.astype(int), K=2)

# 外れ値の追加
X_out = np.vstack([X_clean, rng.multivariate_normal([7.8, -6.0], np.eye(2)*0.3, size=10)])
y_out = np.hstack([y_clean, np.ones(10)])
T_out = to_one_of_k(y_out.astype(int), K=2)

# モデル学習
ls_clean = LeastSquaresClassifier().fit(X_clean, T_clean)
lr_clean = LogisticRegression2Class().fit(X_clean, y_clean)

ls_dirty = LeastSquaresClassifier().fit(X_out, T_out)
lr_dirty = LogisticRegression2Class().fit(X_out, y_out)

# 誤分類率の計算
err_ls_clean = np.mean(ls_clean.predict(X_clean) != y_clean)
err_lr_clean = np.mean(lr_clean.predict(X_clean) != y_clean)
err_ls_dirty = np.mean(ls_dirty.predict(X_out) != y_out)
err_lr_dirty = np.mean(lr_dirty.predict(X_out) != y_out)

print(f"Clean Dataset  - Least Squares Error: {err_ls_clean*100:.1f}%, Logistic Regression Error: {err_lr_clean*100:.1f}%")
print(f"Dirty (Outlier)- Least Squares Error: {err_ls_dirty*100:.1f}%, Logistic Regression Error: {err_lr_dirty*100:.1f}%")

# 線形制約保存定理の数値検証（すべての入力で出力の和が1になること）
grid_x = np.random.uniform(-5, 5, size=(10, 2))
pred_vals = ls_clean.predict_values(grid_x)
row_sums = pred_vals.sum(axis=1)
print(f"Prediction row sums on random inputs: {row_sums}")
assert np.allclose(row_sums, 1.0), "Sum of predictions must always be 1.0!"
print("Linear constraint preservation theorem verified with machine precision!")


---
## まとめと洞察 (Summary & Key Insights)

1. **線形判別関数の幾何学 (5.1.1)**:
   - 重みベクトル $\mathbf{w}$ は決定境界超平面の法線ベクトルであり、バイアス $w_0$ は原点からの符号付き直交距離 $-w_0/\|\mathbf{w}\|$ を決定します。
   - 判別関数の値 $y(\mathbf{x})$ は決定境界からの符号付き直交距離に比例し、直交射影点 $\mathbf{x}_\perp = \mathbf{x} - \frac{y(\mathbf{x})}{\|\mathbf{w}\|} \frac{\mathbf{w}}{\|\mathbf{w}\|}$ を与えます。
2. **多クラス分類と決定領域の凸性 (5.1.2)**:
   - 2クラス分類器を 1対他 や 1対1 で組み合わせると、未分類・曖昧領域（Figure 5.2）が不可避的に発生します。
   - 単一の $K$ クラス判別関数 $y_k(\mathbf{x}) = \mathbf{w}_k^T \mathbf{x} + w_{k0}$ を用いれば、決定領域 $\mathcal{R}_k$ は常に単連結かつ凸集合となり（Figure 5.3）、曖昧領域は完全に排除されます。
3. **1-of-K 符号化と最小二乗法 (5.1.3 & 5.1.4)**:
   - 最小二乗法は擬似逆行列による閉形式解 $\widetilde{\mathbf{W}} = \widetilde{\mathbf{X}}^\dagger \mathbf{T}$ を持ち、ターゲットの線形制約を保存するため出力の和は常に 1 になります。
   - しかし、ガウス雑音仮定の不整合により出力が $[0, 1]$ を逸脱し、特に正しく分類された遠方の外れ値に対して過大なペナルティを課すため、決定境界が激しく回転して誤分類が急増します（Figure 5.4）。
   - この問題は、次節以降で扱う確率的識別モデル（ロジスティック回帰・ソフトマックス回帰）や生成モデルによって解決されます。
